In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import Audio      # βιβιοθήκη για να "παίζουμε" ήχους (μεταξύ άλλων)
from scipy import signal
from scipy.io import wavfile
import time

## Άσκηση - Agent 007: ανίχνευση αριθμού τηλεφώνου

Όταν πληκτρολογούμε ένα τηλεφωνικό νούμερο στο κινητό μας ή σε ένα οποιοδήποτε τηλέφωνο με πλήκτρα, δημιουργούμε για κάθε αριθμό δυο συνημίτονα. Για παράδειγμα, όταν πληκτρολογούμε τον αριθμό $0$, δημιουργούμε ένα άθροισμα δυο συνημιτόνων με συχνότητες $941$ και $1336$ Hz έκαστο, ήτοι

$$ x_0[n] = \cos\Big(2\pi 941 \frac{n}{f_s}\Big) + \cos\Big(2\pi 1336 \frac{n}{f_s}\Big) \tag{6}$$

με $f_s$ τη συχνότητα δειγματοληψίας, η οποία είναι $f_s = 8000$ Hz για όλο το σύστημα. Το πρότυπο αυτό ονομάζεται DTMF - Dual-Tone Multi-Frequency, και ο πίνακας που αντιστοιχεί σε αυτό φαίνεται παρακάτω στον πίνακα του Σχήματος 1.

<p style="text-align: center"><img src="./files/DTMF.jpg"></p>

<p style="text-align:center">Σχήμα 1: Πίνακας Πρότυπου DTMF.</p>

Οι συχνότητες που αντιστοιχούν σε κάθε αριθμό δεν έχουν επιλεγεί τυχαία. Καμιά από τις συχνότητες δεν είναι πολλαπλάσιο κάποιας άλλης, άθροισμα ή διαφορά οποιωνδήποτε δυο συχνοτήτων κλπ. Η συνθήκη αυτή διευκολύνει πολύ τον εντοπισμό των συχνοτήτων (και άρα τον αριθμό). 

Για τους σκοπούς μας, κάθε άθροισμα συνημιτόνων (δηλ. κάθε τηλεφωνικός *τόνος*) διαρκεί $0.5$ δευτερόλεπτα, ενώ υπάρχει μια παύση (σιωπή) μεταξύ των τόνων, διάρκειας $0.1$ δευτερολέπτων. Στο διακριτό χρόνο, η παύση δεν είναι τίποτε άλλο από ένα διάνυσμα γεμάτο μηδενικά, κατάλληλης διάρκειας δειγμάτων. Υποθέτουμε ότι ο αριθμός που ψάχνουμε έχει συντεθεί από διαδοχικές συνενώσεις τόνου + σιωπής διάρκειας $0.5 + 0.1 = 0.6$ δευτερολέπτων ο καθένας. Άρα ένα $10$ψήφιο τηλεφωνικό σήμα θα έχει διάρκεια $6$ δευτερόλεπτα. Το σκεπτικό μας είναι να χωρίσουμε το ολόκληρο το σήμα τηλεφωνικού αριθμού σε "κομμάτια" (frames) των $0.6$ δευτερολέπτων, χωρίς αυτά να επικαλύπτονται μεταξύ τους, και να ελέγχουμε σε κάθε frame αν υπάρχει κάποιος τόνος από τους παραπάνω. 

**Σκοπός αυτής της άσκησης είναι να σχεδιάσετε ένα σύστημα ζωνοπερατών (bandpass) φίλτρων (δηλ. συστημάτων) που θα φιλτράρει ένα frame κάθε φορά, και θα ελέγχει ποιό ζεύγος εξόδων φίλτρων έχει περισσότερη πληροφορία σε σχέση με τις υπόλοιπες, ελέγχοντας την ενέργειά καθεμιάς σε σχέση με ένα κατώφλι (threshold). Αυτό το ζεύγος εξόδων - που αντιστοιχεί σε ένα ζεύγος ζωνοπερατών φίλτρων - θα μας υποδείξει ποιά φίλτρα χρησιμοποιήθηκαν και άρα ποιές συχνότητες επέτρεψαν αυτά να περάσουν. Αυτές οι συχνότητες θα πρέπει να είναι συχνότητες που υπάρχουν στον Πίνακα παραπάνω**.

Το σύστημα ζωνοπερατών φίλτρων θα σχεδιαστεί από ένα βασικό (ιδανικό) χαμηλοπερατό (lowpass) φίλτρο το οποίο έχει απόκριση σε συχνότητα στο διάστημα $(-\pi, \pi]$ ως

$$ H_{lp}(e^{j\omega}) = \left\{\begin{array}{lll}
1, & \displaystyle - \omega_c < \omega < \omega_c \\
 & \\
0, & \displaystyle \omega_c \leq |\omega| \leq \pi
\end{array}\right. \tag{7}$$

με $\omega_c = \pi/200$ (η οποία αντιστοιχεί σε $f_c = 20$ Hz στο συνεχή χρόνο).  **Το σκεπτικό είναι πως αντί να σχεδιάσετε πολλά διαφορετικά ζωνοπερατά φίλτρα, μπορείτε να χρησιμοποιήσετε την ιδιότητα της μετατόπισης στη συχνότητα του μετασχ. Fourier διακριτού χρόνου ώστε να μετατοπίζετε αυτό το βασικό φίλτρο γύρω από τις κατάλληλες συχνότητες $\pm \omega_0$ που σας ενδιαφέρουν, και να δημιουργείτε έτσι τα εκάστοτε ζωνοπερατά φίλτρα**. Παρατηρήστε ότι 

$$ 2\cos(\omega_0n) h[n] \longleftrightarrow H(e^{j(\omega - \omega_0)}) + H(e^{j(\omega + \omega_0)}) \tag{8}$$

Το σύστημά μας αποτελείται από *ιδανικά* φίλτρα, οπότε αρκεί αυτά να έχουν άρτια συμμετρία στο χώρο της συχνότητας. Το διάγραμμα του Σχήματος 2 σας δείχνει πως δουλεύει το σύστημά σας για το πρώτο frame διάρκειας $0.6$ s, που αντιστοιχεί στον αριθμό $6$.

<p style="text-align: center"><img src="./files/digitRec.png" width="1400"></p>

<p style="text-align:center">Σχήμα 2: Διάγραμμα λειτουργίας συστήματος ανίχνευσης αριθμού για το πρώτο frame.</p>

Ως παράδειγμα σας δίνουμε ένα αρχείο ήχου που περιέχει τόνους από έναν αριθμό ενός κινητού τηλεφώνου. **Τελικός σκοπός σας είναι να εντοπίσετε τον αριθμό**. Μπορείτε να χρησιμοποιήσετε τις παρακάτω εντολές για να το ακούσετε.

    fs, s = wavfile.read('./cell_num.wav')    # διαβάζουμε το συνοδευτικό wav αρχείο
    s = s / (2**15)                           # κανονικοποίηση
    Audio(s, rate=fs)
    
Σας δίνονται επίσης **τρεις συναρτήσεις που πρέπει να συμπληρώσετε και οι οποίες εκτελούν συνολικά την ανίχνευση του αριθμού**. Ακολουθήστε τις παρακάτω οδηγίες για κάθε συνάρτηση.


**1. makeFB**

Σε αυτήν την συνάρτηση θα φτιάξετε το σύστημα ζωνοπερατών φίλτρων $h_i[n]$. Υπενθυμίζεται ότι σκοπός είναι να φτιάξετε τα φίλτρα αυτά χρησιμοποιώντας μόνο το χαμηλοπερατό φίλτρο της Σχέσης (7) και κατάλληλες πράξεις με αυτό. Το πλήθος τους σύμφωνα με το πρότυπο DTMF πρέπει να είναι επτά ($7$), ένα για κάθε μια συχνότητα, αν και στον κώδικα αυτό δίνεται ως μεταβλητή από το χρήστη για περισσότερη γενικότητα. Η συνάρτηση αυτή σας επιστρέφει έναν πίνακα που η κάθε στήλη του περιέχει $201$ δείγματα της κρουστικής απόκρισης $h[n]$ από ένα φίλτρο. Π.χ. η $3$η στήλη του πίνακα περιέχει την κρουστική απόκριση του ζωνοπερατού φίλτρου που έχει σαν κέντρο του τη συχνότητα $2\pi 852/fs$ rad/sample. Άρα ο πίνακας που κατασκευάζετε και επιστρέφει η συνάρτηση πρέπει να έχει διαστάσεις $201 \times 7$.


(α') Συμπληρώστε στη γραμμή $20$ παρακάτω τη συχνότητα αποκοπής $\omega_c$ του βασικού χαμηλοπερατού φίλτρου που σας δίνεται παραπάνω.


(β') Βρείτε αρχικά στο χαρτί ποιό είναι το σήμα στο χρόνο στο οποίο αντιστοιχεί το χαμηλοπερατό φίλτρο που αναφέρεται παραπάνω στη Σχέση (7) και **παραδώστε την απάντησή σας παρακάτω**. Εκφράστε την απάντησή σας ως συνάρτηση της πολύ γνωστής σας συνάρτησης $\textrm{sinc}$.

### Απάντηση:

Το χαμηλοπερατό φίλτρο

$$ H_{lp}(e^{j\omega}) = \left\{\begin{array}{lll}
1, & \displaystyle - \omega_c < \omega < \omega_c \\
 & \\
0, & \displaystyle \omega_c \leq |\omega| \leq \pi
\end{array}\right. $$

με βάση τα γνωστά ζεύγη Μετασχηματισμού Fourier Διακριτού Χρόνου, γνωρίζουμε ότι αντιστοιχεί στο εξής σήμα στο χρόνο: 

$$h_{lp}[n] = \frac{\sin(\omega_c n)}{\pi n}\Longrightarrow h_{lp}[n] = \frac{\omega_c}{\pi}\mathrm{sinc}\left(\frac{\omega_c n}{\pi}\right)$$

(γ') Γράψτε την έκφραση που βρήκατε στη γραμμή $34$ της συνάρτησης, για να δημιουργήσετε $201$ δείγματα του βασικού χαμηλοπερατού σας φίλτρου. Υπάρχει η συνάρτηση $\textrm{sinc}$ της Python η οποία θα σας φανεί *πολύ πολύ* χρήσιμη.

(δ') Βρείτε το μετασχηματισμό Fourier του σήματος 
$$ h[n] = 2\cos(\omega_0 n)h_{lp}[n] \tag{9}$$

στο χαρτί σας. Εκφράστε τον ως συνάρτηση του μετασχ. Fourier του $h_{lp}[n]$, δηλ. του $H_{lp}(e^{j\omega})$. Τι παρατηρείτε; **Απαντήστε παρακάτω**.

### Απάντηση:

Είναι: 

$$h[n] = 2\cos(\omega_0 n)h_{lp}[n] = (e^{j\omega_0 n}+e^{-j\omega_0 n})h_{lp}[n] \Longrightarrow h[n] = e^{j\omega_0 n} h_{lp}[n] + e^{-j\omega_0 n} h_{lp}[n]$$

Ο μετασχηματισμός Fourier του σήματος $h[n]$, βάσει των ιδιοτήτων της γραμμικότητας και της μετατόπισης στη συχνότητα, είναι ο ακόλουθος:

$$H(e^{j\omega}) = H_{lp}(e^{j(\omega-\omega_0)})+H_{lp}(e^{j(\omega+\omega_0)})$$

Η απόκριση συχνότητας αποτελείται από δυο χαμηλοπερατά φίλτρα γύρω από τις συχνότητες $\pm \omega_0$, δηλ. έχουμε κατασκευάσει ένα ζωνοπερατό φίλτρο!

(ε') Μπορείτε τώρα να συμπληρώστε τη γραμμή $39$ της συνάρτησης δημιουργώντας έτσι τα ζωνοπερατά σας φίλτρα $h_i[n]$ επιλέγοντας κατάλληλες $\omega_0$ για το καθένα. Η υλοποίησή σας πρέπει να γίνει στο πεδίο του χρόνου - μια απλή πράξη.

(ζ') Τέλος, στη γραμμή $41$, κανονικοποιήστε το κάθε φίλτρο που δημιουργείτε, απλά διαιρώντας το με το άθροισμα των τιμών των δειγμάτων του, 

$$\sum_{n=-\infty}^{+\infty}h_i[n] \tag{10}$$

Η συνάρτηση $\textrm{sum}$ της numpy θα σας φανεί χρήσιμη.

In [ ]:
def makeFB(fr):
    # Δημιουργεί μια συστοιχία ζωνοπερατών φίλτρων. Κάθε φίλτρο έχει εύρος ζώνης 40 Hz.
    # Τα φίλτρα είναι κεντραρισμένα στις συχνότητες που καθορίζονται από το fr
    # 
    # 
    # Για παράδειγμα, για το πρότυπο DTMF:
    # makeFB([697,770,852,941,1209,1336,1477]);

    # Συχνότητα δειγματοληψίας
    fs = 8000

    # Συχνότητα αποκοπής του χαμηλοπερατού φίλτρου (σε Hz)
    fc = 20

    # Γραμμή 20 παρακάτω: Συχνότητα αποκοπής του χαμηλοπερατού φίλτρου (σε rad/sample)
    wc = 2*np.pi*fc/fs

    # Μήκος του διανύσματος εισόδου
    L = len(fr)

    # Δημιουργία 201 δειγμάτων φίλτρου
    n = np.arange(-100,101)
    # Δέσμευση μνήμης
    h = np.zeros([len(n), L])

    # Βασικό χαμηλοπερατό φίλτρο

    # Αποθηκεύστε στο hlp την εξίσωση του χαμηλοπερατού φίλτρου στο πεδίο του χρόνου
    # Γραμμή 34 παρακάτω: Χρησιμοποιήστε τη συνάρτηση "sinc" και το διάνυσμα "n" παραπάνω
    hlp = wc/np.pi * np.sinc(wc*n/np.pi)

    # Δημιουργία των ζωνοπερατών φίλτρων
    for i in range(0,L):
        # Γραμμή 39 παρακάτω: Αποθηκεύστε στο h[:,i] το i-οστό ζωνοπερατό φίλτρο συχνότητας fr[i]
        h[:,i] = 2*np.cos(2*np.pi*fr[i]/fs*n)* hlp
        # Γραμμή 41 παρακάτω: Διαιρέστε το φίλτρο σας με το άθροισμα των συντελεστών του
        h[:,i] = h[:,i] / np.sum(h[:,i])

    return h

**2. dDTMF**

Η συνάρτηση αυτή δέχεται ως όρισμα ένα σήμα τηλεφωνικού αριθμού και εντοπίζει τον τηλεφωνικό αριθμό που μεταφέρει το σήμα. Σημειώστε ότι γίνεται κλήση της συνάρτησης $\textrm{makeFB}$. Η συνάρτηση $\textrm{dDTMF}$ σας επιστρέφει έναν πίνακα δυο στηλών ο οποίος περιέχει τις συχνότητες που μεταφέρει το σήμα, καθώς και έναν πίνακα χαρακτήρων που περιέχει τον αριθμό που ανιχνεύτηκε. Συγκεκριμένα:

(α') Γραμμή $24$: Μετατρέψτε τη διάρκεια κάθε τόνου από δευτερόλεπτα σε δείγματα. Θα σας χρειαστεί η συχνότητα δειγματοληψίας $\textrm{fs}$.

(β') Γραμμή $30$: Κάντε το ίδιο για τις σιωπές.

(γ') Γραμμή $33$: (απλό σχόλιο) Στη γραμμή αυτή, βρίσκουμε πόσα frames τόνου+σιωπής υπάρχουν σε όλο το τηλεφωνικό σήμα. Απλώς κατανοήστε γιατί δουλεύει σωστά αυτή η εντολή.

(δ') Γραμμή $45$: (απλο σχόλιο) Στη γραμμή αυτή, ο βρόχος επανάληψης διατρέχει το σήμα εισόδου ανά sh δείγματα και αποθηκεύει κάθε φορά στο διάνυσμα fr ένα κομμάτι διάρκειας όσο είναι *μόνο* η διάρκεια του τόνου (st), η οποία θεωρούμε ότι προηγείται της σιωπής πάντα.


(ε') Γραμμή $51$: Υπολογίστε την ενέργεια του κάθε κομματιού που δεσμεύεται, με βάση τη γνωστή σας σχέση

$$E = \sum_{n=-\infty}^{+\infty}x^2[n] \tag{11}$$

Ξανά, η συνάρτηση $\textrm{sum}$ της numpy θα σας φανεί χρήσιμη.

(στ') Γραμμή $55$: (απλό σχόλιο) Σε αυτή τη γραμμή, ο βρόχος επανάληψης διατρέχει όλα τα ζωνοπερατά φίλτρα που έχουμε φτιάξει στον πίνακα h και κάνει συνέλιξη καθενός από αυτά με το σήμα που βρίσκεται στο διάνυσμα fr. Προσέξτε ότι τα διαφορετικά φίλτρα βρίσκονται σε στήλες στον πίνακα h. Η διαδικασία αυτή υλοποιεί ακριβώς την έξοδο ενός ΓΧΑ συστήματος δεδομένης μιας εισόδου του (a.k.a συνέλιξη ).

(ζ΄) Γραμμή $58$: Κάνετε συνέλιξη του σήματος με το j-οστό ζωνοπερατό φίλτρο ώστε να βρείτε την έξοδο του φίλτρου. Η συνάρτηση $\textrm{conv}$ θα σας χρειαστεί σίγουρα.

(η') Γραμμή $61$: Υπολογίστε την ενέργεια του παραπάνω σήματος εξόδου με τη γνωστή σχέση της ενέργειας.

(θ') Γραμμή $64$: Πρέπει να θέσετε ένα όριο ώστε να αναγνωρίζετε πότε ένα ζωνοπερατό φίλτρο "έχει πιάσει" συχνότητα που περιέχεται στο σήμα fr. Σας προτείνεται ένα τέτοιο κριτήριο μέσα στον κώδικα ως σχόλιο.

(ι') Γραμμές $79-$τέλος: Με βάση τον Πίνακα 2, συμπληρώστε τα ζεύγη συχνοτήτων για κάθε αριθμό.


In [ ]:
def dDTMF(x):
    #  Δεδομένου ενός σήματος x που περιέχει έναν τηλεφωνικό αριθμό σύμφωνα με
    #  το πρότυπο DTMF, επιστρέφει τις συχνότητες που υπάρχουν
    #  στο σήμα και τον αριθμό σε ένα διάνυσμα χαρακτήρων

    #  Συχνότητες DTMF
    DTMFfr = [697, 770, 852, 941, 1209, 1336, 1477]

    #  Μήκος του διανύσματος DTMFfr
    L = len(DTMFfr)

    #  Κατασκευή του πίνακα συχνοτήτων
    h = makeFB(DTMFfr)

    #  Συχνότητα δειγματοληψίας
    fs = 8000

    #  Διάρκεια τόνου σε sec
    d = 0.5

    #  Γραμμή 24 παρακάτω: Διάρκεια τόνου σε δείγματα
    ds = int(d*fs)

    #  Διάρκεια σιωπής σε sec
    sil = 0.1

    #  Γραμμή 30 παρακάτω: Διάρκεια σιωπής σε δείγματα
    sils = sil*fs

    #  Γραμμή 33 παρακάτω: Πόσα πλαίσια "τόνος+σιωπή" χωρούν σε όλο το σήμα;
    Lt = round((len(x) + sils) / (ds+sils))

    #  Μήκος πλαισίου
    st = np.arange(0, ds)
    
    #  Διάρκεια τόνου και σιωπής
    sh = ds + sils

    #  Αρχικοποίηση
    Fvec = np.zeros((Lt, 2))

    #  Γραμμή 45 παρακάτω: Για κάθε πλαίσιο...
    for i in range(0, Lt): 
        
        # Πάρτε το αντίστοιχο τμήμα από το σήμα...	
        fr = x[st.astype(int)]

        # Γραμμή 51 παρακάτω: Υπολογίστε την ενέργεια του πλαισίου fr
        En = np.sum(fr**2)

        k = 0
        # Γραμμή 55 παρακάτω:
        for j in range(0,L):
            
            #  Γραμμή 58 παρακάτω: Φιλτράρετε το τμήμα fr με την np.convolve 
            # και το φίλτρο h[:,j] (χρησιμοποιήστε την επιλογή 'same')
            sc = np.convolve(fr , h[:,j], 'same')
    
            # Γραμμή 61 παρακάτω: Υπολογίστε την ενέργεια μετά το φιλτράρισμα
            S = np.sum(sc**2)

            # Γραμμή 64 παρακάτω: Κανονικοποιήστε το S με το En
            R = S/En

            # Γραμμή 67 παρακάτω: Βρείτε ένα κατώφλι (2 φορές η ενέργεια 
            # του αντίστοιχου φίλτρου h[:,j] δουλεύει καλά)
            if R > 2*np.sum(h[:,j]**2):
                Fvec[i,k] = DTMFfr[j]
                k = k + 1

        st = st + sh;  # μετακίνηση πλαισίου

    # Ανίχνευση αριθμού
    [M,N] = np.shape(Fvec)
    number = ""
    # Γραμμή 79 παρακάτω:
    for i in range(0,M):
        if Fvec[i,0] == 697:
            if Fvec[i,1] == 1209:
                number = number+ '1'
            elif Fvec[i,1] == 1336:
                number = number + '2'
            elif Fvec[i,1] == 1477:
                number = number + '3'

        elif Fvec[i,0] == 770:
            if Fvec[i,1] == 1209:
                number = number + '4'
            elif Fvec[i,1] == 1336:
                number = number +'5'
            elif Fvec[i,1] == 1477:
                number = number +'6'

        elif Fvec[i,0] == 852:
            if Fvec[i,1] == 1209:
                number = number + '7'
            elif Fvec[i,1] == 1336:
                number = number + '8'
            elif Fvec[i,1] == 1477:
                number = number + '9'

        elif Fvec[i,0] == 941:
            if Fvec[i,1] == 1209:
                number = number + '*'
            elif Fvec[i,1] == 1336:
                number = number + '0'
            elif Fvec[i,1] == 1477:
                number = number + '#'

    return Fvec, number


Ας δοκιμάσουμε τις συναρτήσεις που φτιάξαμε κι ας δούμε τι αριθμός προκύπτει!

In [ ]:
fs, s = wavfile.read('./cell_num.wav'); 
s = s/(2**15) 
Audio(s, rate=fs)             # ακούμε! :)

[Freq, Num] = dDTMF(s)

print(f"Ο αριθμός που βρήκαμε είναι ο {Num}")

**3. call**

Η συνάρτηση call σας δίνεται απλά για έλεγχο. Με αυτήν την συνάρτηση, μπορείτε να δημιουργήσετε το δικό σας ήχο από οποιοδήποτε τηλεφωνικό νούμερο, να το ακούσετε, και αν θέλετε να το ανιχνεύσετε με τη βοήθεια των συναρτήσεων που γράψατε πριν, ώστε να βεβαιωθείτε ότι δουλεύει σωστά ο κώδικά σας πριν τον χρησιμοποιήσετε για το μυστικό αριθμό που σας δίνεται. Για παράδειγμα, μπορείτε να γράψετε

    x = call([2 8 1 0 3 9 3 5 3 3]);
    [Freq, Num] = dDTMF(x);

και στη μεταβλητή Num θα πρέπει να έχετε τον αριθμό $2810393533$. Για να δουλέψει αυτή η συνάρτηση, πρέπει να συμπληρώσετε μέσα τις συχνότητες του Πίνακα 2 στις γραμμές $27-50$.

In [ ]:
def call(tel):
    #  Δημιουργεί ένα σήμα που αντιστοιχεί στον τηλεφωνικό αριθμό
    #  που καθορίζεται από την είσοδο tel σύμφωνα με το πρότυπο DTMF
    #  Για παράδειγμα: x = call([6 9 7 1 1 1 1 1 1 1]);

    fs = 8000

    d = 0.5
    ds = d*fs
    sil = 0.1
    sils = sil*fs

    Lt = len(tel)
    D = d*Lt+sil*(Lt-1)
    
    Ds = int(D*fs)  # σε δείγματα

    x = np.zeros((Ds,1))
    st = np.arange(0,ds)
    sh = ds+sils
    n = np.arange(0,ds)

    # Γραμμή 27 παρακάτω:
    for i in range(0,Lt):
        st=st.astype(int)
        if(tel[i]==1):
            x[st, 0] = np.cos(2*np.pi*697/fs*n)+np.cos(2*np.pi*1209/fs*n)
        elif(tel[i]==2):
            x[st, 0] = np.cos(2*np.pi*697/fs*n)+np.cos(2*np.pi*1336/fs*n)
        elif(tel[i]==3):
            x[st, 0] = np.cos(2*np.pi*697/fs*n)+np.cos(2*np.pi*1477/fs*n)
        elif(tel[i]==4):
            x[st, 0] = np.cos(2*np.pi*770/fs*n)+np.cos(2*np.pi*1209/fs*n)
        elif(tel[i]==5):
            x[st, 0] = np.cos(2*np.pi*770/fs*n)+np.cos(2*np.pi*1336/fs*n)
        elif(tel[i]==6):
            x[st, 0] = np.cos(2*np.pi*770/fs*n)+np.cos(2*np.pi*1477/fs*n)
        elif(tel[i]==7):
            x[st, 0] = np.cos(2*np.pi*852/fs*n)+np.cos(2*np.pi*1209/fs*n)
        elif(tel[i]==8):
            x[st, 0] = np.cos(2*np.pi*852/fs*n)+np.cos(2*np.pi*1336/fs*n)
        elif(tel[i]==9):
            x[st, 0] = np.cos(2*np.pi*852/fs*n)+np.cos(2*np.pi*1477/fs*n)
        else:  # 0
            x[st, 0] = np.cos(2*np.pi*941/fs*n)+np.cos(2*np.pi*1336/fs*n)

        st = st+sh

    # Κλιμάκωση:
    x = x / (2**(16 - 1))

    return x


Ας τη δοκιμάσουμε!

In [ ]:

x = call([6,9,0,0,0,0,1,4,8,5])
fs = 8000

Audio(x.T, rate=fs)             # ακούμε! :)

---
---